# RTC-SDD | 00 — CPU Research Setup

**Purpose:** validate the approved private RTCFake dataset, the shared research package, and the pinned official baseline source. **No GPU required; no model training or Fairseq installation.**

**Inputs:** (1) a private Kaggle dataset containing this repository; (2) `rtcfake-packed-private`. Internet ON for Python 3.10 and official source checkout. Run top-to-bottom in a fresh CPU notebook. If a stale project exists in `/kaggle/working`, the code stops rather than silently using old files.


In [ ]:
from pathlib import Path
from shutil import copytree
import hashlib
import shutil
import sys

INPUT = Path("/kaggle/input")
ROOT = Path("/kaggle/working/RTC-SDD-Research")

sources = [
    p.parent for p in INPUT.rglob("pyproject.toml")
    if (p.parent / "src/rtcsdd/data/audio.py").is_file()
    and (p.parent / "scripts/setup/verify_setup.py").is_file()
]
assert len(sources) == 1, f"Expected one updated research code dataset, got {len(sources)}"
SOURCE = sources[0]

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

if ROOT.exists():
    for rel in (
        "pyproject.toml", "src/rtcsdd/data/audio.py",
        "scripts/setup/bootstrap_official.py", "scripts/setup/verify_setup.py",
        "third_party/official_rtc_sdd.pin.json",
    ):
        assert (ROOT / rel).is_file() and digest(ROOT / rel) == digest(SOURCE / rel), (
            f"Stale /kaggle/working copy: {rel}. Restart clean or back up and remove it."
        )
else:
    copytree(SOURCE, ROOT)

matches = list(INPUT.rglob("train_offline.pack"))
assert len(matches) == 1, f"Expected one RTCFake private dataset, got {len(matches)}"
DATA_ROOT = matches[0].parent

print("Project:", ROOT)
print("Dataset:", DATA_ROOT)
print("Python:", sys.version.split()[0])
print("Free working disk: %.2f GiB" % (shutil.disk_usage("/kaggle/working").free / 1024**3))


## 1 — Install and test the shared package

This uses the notebook's existing CPU Python. Unit tests use **synthetic** data; they are not a complete real-audio audit.


In [ ]:
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{ROOT}[dev]"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, check=True)
print("[PASS] Infrastructure unit tests")


## 2 — Pin upstream code and Python 3.10 (CPU)

The Python 3.10 environment is **empty**; the old CUDA/Fairseq dependencies are **not** installed in this notebook. The official source stays under `third_party/`, pinned and unmodified.


In [ ]:
import shutil

assert shutil.which("uv"), "Kaggle environment needs uv"
ENV = Path("/kaggle/working/rtcsdd-baseline-py310")
PYTHON310 = ENV / "bin/python"

if not PYTHON310.is_file():
    subprocess.run(["uv", "python", "install", "3.10"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.10", str(ENV)], check=True)

subprocess.run(
    [sys.executable, str(ROOT / "scripts/setup/bootstrap_official.py")],
    cwd=ROOT, check=True,
)
subprocess.run(
    [sys.executable, str(ROOT / "scripts/setup/install_training_env.py"),
     "--python", str(PYTHON310)],
    cwd=ROOT, check=True,
)
print("[PASS] Pinned source and check-only environment plan")


## 3 — Final CPU verification

Checks tests, dataset-to-pack membership, 16 representative WAV decodes, pinned upstream checkout, and check-only training requirements. **Success does not mean XLS-R can train yet.**


In [ ]:
subprocess.run(
    [sys.executable, str(ROOT / "scripts/setup/verify_setup.py"),
     "--dataset-root", str(DATA_ROOT), "--python310", str(PYTHON310)],
    cwd=ROOT, check=True,
)
print("[PASS] CPU setup complete. GPU training environment remains unverified.")


## Next: GPU smoke notebook (separate)

Complete official Fairseq/torch dependency resolution, verify the approved XLS-R checkpoint (source and SHA256), compare audio loader parity, test one-GPU forward/backward, memory, resume, and inference. **No full training until those pass.**

The code/configs/scripts live in GitHub, not only in this notebook. See `docs/setup.md` and `docs/experiments.md`.
